# Z.AI receipt OCR

This notebook sends one manually selected receipt image URL to `zai-sdk`, runs `client.layout_parsing.create(...)` first, then sends the parsed markdown to `client.chat.completions.create(...)` for structured extraction, prints the image URL, shows the image, reports token usage, computes cost with the requested formula, and returns structured JSON.

In [5]:
from __future__ import annotations

import json
import os
from pathlib import Path

from IPython.display import Image, display
from zai import ZaiClient


In [6]:
def load_env_file(env_path: Path = Path('.env')) -> None:
    if not env_path.exists():
        return

    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue

        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip("\"").strip("'")
        if key and key not in os.environ:
            os.environ[key] = value


def build_extraction_prompt(layout_markdown: str) -> str:
    return f'''Extract the receipt fields from the layout parsing markdown below.
Return only valid JSON in this shape:
{{
  "merchant_name": "",
  "invoice_date": "",
  "currency": "",
  "total_charge": null
}}

Layout parsing markdown:
{layout_markdown}
'''.strip()


def parse_json_content(content: object) -> dict[str, object]:
    if content is None:
        return {}

    if isinstance(content, list):
        content = ''.join(
            part.get('text', '') if isinstance(part, dict) else str(part)
            for part in content
        )

    if not isinstance(content, str):
        return {}

    try:
        payload = json.loads(content)
    except json.JSONDecodeError:
        return {}

    return payload if isinstance(payload, dict) else {}


def github_raw_url(filename: str) -> str:
    return f'https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/charges/{filename}'


def price_per_million(model_name: str) -> dict[str, float]:
    price_table = {
        'glm-ocr': {'input': 0.03, 'output': 0.03},
        'glm-5': {'input': 1.0, 'output': 3.2},
    }
    normalized_name = model_name.lower()
    if normalized_name not in price_table:
        raise ValueError(f'No pricing configured for model: {model_name}')

    return price_table[normalized_name]


def process_file(client: ZaiClient, image_url: str, layout_model_name: str, chat_model_name: str) -> dict[str, object]:
    layout_pricing = price_per_million(layout_model_name)
    chat_pricing = price_per_million(chat_model_name)

    print(f'Image URL: {image_url}')
    display(Image(url=image_url))

    layout_response = client.layout_parsing.create(
        model=layout_model_name,
        file=image_url,
    )
    layout_payload = layout_response.to_dict(mode='json', exclude_none=True)
    layout_usage = layout_payload.get('usage') or {}
    layout_markdown = layout_payload.get('md_results') or ''
    layout_input_tokens = int(layout_usage.get('prompt_tokens') or 0)
    layout_output_tokens = int(layout_usage.get('completion_tokens') or 0)
    layout_tokens = int(layout_usage.get('total_tokens') or 0)
    if not layout_tokens:
        layout_tokens = layout_input_tokens + layout_output_tokens

    if not layout_markdown:
        raise ValueError('Layout parsing returned no markdown content.')

    extraction_response = client.chat.completions.create(
        model=chat_model_name,
        temperature=0,
        response_format={'type': 'json_object'},
        messages=[
            {
                'role': 'user',
                'content': build_extraction_prompt(layout_markdown),
            },
        ],
    )

    usage = extraction_response.usage
    extraction_input_tokens = int(usage.prompt_tokens or 0) if usage else 0
    extraction_output_tokens = int(usage.completion_tokens or 0) if usage else 0
    extraction_tokens = int(usage.total_tokens or 0) if usage else 0
    if not extraction_tokens:
        extraction_tokens = extraction_input_tokens + extraction_output_tokens

    total_input_tokens = layout_input_tokens + extraction_input_tokens
    total_output_tokens = layout_output_tokens + extraction_output_tokens
    total_tokens = layout_tokens + extraction_tokens

    layout_input_cost_usd = layout_input_tokens / 1_000_000 * layout_pricing['input']
    layout_output_cost_usd = layout_output_tokens / 1_000_000 * layout_pricing['output']
    chat_input_cost_usd = extraction_input_tokens / 1_000_000 * chat_pricing['input']
    chat_output_cost_usd = extraction_output_tokens / 1_000_000 * chat_pricing['output']
    input_cost_usd = layout_input_cost_usd + chat_input_cost_usd
    output_cost_usd = layout_output_cost_usd + chat_output_cost_usd
    cost_usd = input_cost_usd + output_cost_usd

    structured_data = parse_json_content(extraction_response.choices[0].message.content)

    raw_total_charge = structured_data.get('total_charge')
    try:
        total_charge = float(str(raw_total_charge).replace(',', '')) if raw_total_charge is not None else None
    except (TypeError, ValueError):
        total_charge = None

    print(f'Layout parsing input tokens: {layout_input_tokens}')
    print(f'Layout parsing output tokens: {layout_output_tokens}')
    print(f'Layout parsing input cost (USD): ${layout_input_cost_usd:.8f}')
    print(f'Layout parsing output cost (USD): ${layout_output_cost_usd:.8f}')
    print(f'Chat completion input tokens: {extraction_input_tokens}')
    print(f'Chat completion output tokens: {extraction_output_tokens}')
    print(f'Chat completion input cost (USD): ${chat_input_cost_usd:.8f}')
    print(f'Chat completion output cost (USD): ${chat_output_cost_usd:.8f}')
    print(f'Total input tokens: {total_input_tokens}')
    print(f'Total output tokens: {total_output_tokens}')
    print(f'Total input cost (USD): ${input_cost_usd:.8f}')
    print(f'Total output cost (USD): ${output_cost_usd:.8f}')
    print(f'Total tokens: {total_tokens}')
    print(f'Total cost (USD): ${cost_usd:.8f}')
    print(f'Total charge: {"not found" if total_charge is None else f"${total_charge:.2f}"}')
    print()

    return {
        'image_url': image_url,
        'layout_input_tokens': layout_input_tokens,
        'layout_output_tokens': layout_output_tokens,
        'layout_tokens': layout_tokens,
        'chat_input_tokens': extraction_input_tokens,
        'chat_output_tokens': extraction_output_tokens,
        'chat_tokens': extraction_tokens,
        'total_input_tokens': total_input_tokens,
        'total_output_tokens': total_output_tokens,
        'total_tokens': total_tokens,
        'input_cost_usd': input_cost_usd,
        'output_cost_usd': output_cost_usd,
        'cost_usd': cost_usd,
        'total_charge': total_charge,
        'layout_markdown': layout_markdown,
        'structured_data': structured_data,
    }


In [7]:
AVAILABLE_INPUTS = [
    github_raw_url('1077-receipt.jpg'),
    github_raw_url('invoice_cn_taxi.jpg'),
    github_raw_url('invoice_cn_vat.jpg'),
]

LAYOUT_MODEL_NAME = 'glm-ocr'
CHAT_MODEL_NAME = 'glm-5'

load_env_file()
api_key = os.getenv('ZAI_API_KEY') or os.getenv('API_KEY')
if not api_key:
    raise ValueError('Set ZAI_API_KEY or API_KEY in the environment or .env before running this notebook.')

client = ZaiClient(api_key=api_key)


In [8]:
process_file(client, AVAILABLE_INPUTS[0], LAYOUT_MODEL_NAME, CHAT_MODEL_NAME)

Image URL: https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/charges/1077-receipt.jpg


Layout parsing input tokens: 897
Layout parsing output tokens: 234
Layout parsing input cost (USD): $0.00002691
Layout parsing output cost (USD): $0.00000702
Chat completion input tokens: 303
Chat completion output tokens: 1587
Chat completion input cost (USD): $0.00030300
Chat completion output cost (USD): $0.00507840
Total input tokens: 1200
Total output tokens: 1821
Total input cost (USD): $0.00032991
Total output cost (USD): $0.00508542
Total tokens: 3021
Total cost (USD): $0.00541533
Total charge: $69.79



{'image_url': 'https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/charges/1077-receipt.jpg',
 'layout_input_tokens': 897,
 'layout_output_tokens': 234,
 'layout_tokens': 1131,
 'chat_input_tokens': 303,
 'chat_output_tokens': 1587,
 'chat_tokens': 1890,
 'total_input_tokens': 1200,
 'total_output_tokens': 1821,
 'total_tokens': 3021,
 'input_cost_usd': 0.00032991,
 'output_cost_usd': 0.00508542,
 'cost_usd': 0.00541533,
 'total_charge': 69.79,
 'layout_markdown': 'NAANCHING\n\n103 MONTGOMERRY ST\n\nJERSEY CITY, NJ 07302\n\n2019840709\n\n<div align="center">\n\n# ORDER: SECOND FLOOR 19 Dine-in\n\n</div>\n\nCashier: Kiran\n\n24-Mar-2019 7:14:55P\n\n1 Chicken Lollipop\n\n$9.00\n\nMed $0.00\n\n1 Thai Fried Rice\n\n$12.00\n\nVegetables $0.00\n\nMed $0.00\n\n1 Chicken Tikka Masala $16.00\n\nMILD PLEASE\n\n1 Naan $3.00\n\n1 Vegetable Biryani $12.00\n\nMAKE IT MILD PLEASE\n\n1 Mango Lassi $4.00\n\nSubtotal $56.00\n\nTax $3.71\n\nService Charge (18.0%) $10.08\n\nTotal $69.79',

In [9]:
process_file(client, AVAILABLE_INPUTS[1], LAYOUT_MODEL_NAME, CHAT_MODEL_NAME)

Image URL: https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/charges/invoice_cn_taxi.jpg


Layout parsing input tokens: 833
Layout parsing output tokens: 103
Layout parsing input cost (USD): $0.00002499
Layout parsing output cost (USD): $0.00000309
Chat completion input tokens: 200
Chat completion output tokens: 1792
Chat completion input cost (USD): $0.00020000
Chat completion output cost (USD): $0.00573440
Total input tokens: 1033
Total output tokens: 1895
Total input cost (USD): $0.00022499
Total output cost (USD): $0.00573749
Total tokens: 2928
Total cost (USD): $0.00596248
Total charge: not found



{'image_url': 'https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/charges/invoice_cn_taxi.jpg',
 'layout_input_tokens': 833,
 'layout_output_tokens': 103,
 'layout_tokens': 936,
 'chat_input_tokens': 200,
 'chat_output_tokens': 1792,
 'chat_tokens': 1992,
 'total_input_tokens': 1033,
 'total_output_tokens': 1895,
 'total_tokens': 2928,
 'input_cost_usd': 0.00022499,
 'output_cost_usd': 0.0057374900000000005,
 'cost_usd': 0.005962480000000001,
 'total_charge': None,
 'layout_markdown': '(1) 发条代码、发票号码及密码采用数码喷\n\n恨\n\n北京市出租汽车专用发票\n\nBEIJING TAXI SPECIAL INVOICE\n\n国家税务局监制\n\nINVOICE\n\n111001481004\n\n82047953\n\n单位\n\nCompany\n\n电话\n\nTel\n\nTaxi No.\n\n证号\n\n机打发票 手写无效\n\n北京市出租汽车\n\n发票专用章\n\n密码\n\nPassword',
 'structured_data': {'merchant_name': '北京市出租汽车专用发票',
  'invoice_date': '',
  'currency': '',
  'total_charge': None}}

In [10]:
process_file(client, AVAILABLE_INPUTS[2], LAYOUT_MODEL_NAME, CHAT_MODEL_NAME)

Image URL: https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/charges/invoice_cn_vat.jpg


Layout parsing input tokens: 2423
Layout parsing output tokens: 681
Layout parsing input cost (USD): $0.00007269
Layout parsing output cost (USD): $0.00002043
Chat completion input tokens: 689
Chat completion output tokens: 1245
Chat completion input cost (USD): $0.00068900
Chat completion output cost (USD): $0.00398400
Total input tokens: 3112
Total output tokens: 1926
Total input cost (USD): $0.00076169
Total output cost (USD): $0.00400443
Total tokens: 5038
Total cost (USD): $0.00476612
Total charge: $38266.00



{'image_url': 'https://raw.githubusercontent.com/ikhwanh/zai-api-call/master/images/charges/invoice_cn_vat.jpg',
 'layout_input_tokens': 2423,
 'layout_output_tokens': 681,
 'layout_tokens': 3104,
 'chat_input_tokens': 689,
 'chat_output_tokens': 1245,
 'chat_tokens': 1934,
 'total_input_tokens': 3112,
 'total_output_tokens': 1926,
 'total_tokens': 5038,
 'input_cost_usd': 0.0007616900000000001,
 'output_cost_usd': 0.004004430000000001,
 'cost_usd': 0.00476612,
 'total_charge': 38266.0,
 'layout_markdown': '![](page=0,bbox=[186, 41, 350, 201])\n\n1100162130\n\n<div align="center">\n\n# 北京增值税专用发票\n\n</div>\n\n全国统一发票监制章\n\n北京\n\n国家税务局监制\n\nNo 09750785\n\n09750785\n\n开票日期： 2017年04月18日\n\n<table class="table table-bordered"><thead><tr><th rowspan="2">购买方</th><td colspan="5">名称：中国科学院自动化研究所<br>纳税人识别号：110108400010945<br>地址、电话：北京市海淀区中关村东路95号 010-62554290<br>开户行及账号：农行北京科院南路支行 11-250101040012109</td><th rowspan="2">密码区</th><td colspan="3">45++<0*2140287&gt;4/62&lt;892&gt;/18<br>/2&lt;534*993&lt;